# Step 1: Gene coordinates

Collapse the GTF transcripts into one row per gene. Main chromosomes only, since the scaffolds have no Hi-C data anyway. This table feeds every later step (histone windows, Hi-C bin overlap, hub labels).

## Config

In [9]:
# paths + constants

from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

GTF = RAW / "gene_annotation" / "dm3.refGene.gtf"
WIG_DIR = RAW / "histone_wig"
HIC_DIR = RAW / "hic_contacts"

BIN_SIZE = 10_000
MIN_DISTANCE_BP = 500_000
SEED = 42
STAGES = ["C1-8", "C9-13", "C14E", "C14L"]
MAIN_CHROMS = ["chr2L", "chr2R", "chr3L", "chr3R", "chr4", "chrX"]
WIG_CYCLE = {"C1-8": "c8", "C9-13": "c12", "C14E": "c14a", "C14L": "c14c"}

assert GTF.exists(), f"missing {GTF}"
assert HIC_DIR.exists() and WIG_DIR.exists()
print("root:", ROOT)

root: C:\Users\Zsyvette\Documents\histonet


## Load and filter transcripts

In [10]:
# main chromosomes only, scaffolds drop out here since they have no Hi-C signal later on

gtf_cols = ["chrom", "src", "feature", "start", "end", "score", "strand", "frame", "attrs"]
gtf = pd.read_csv(GTF, sep="\t", header=None, comment="#", names=gtf_cols)

tx = gtf[gtf["feature"] == "transcript"].copy()
tx["gene_name"] = tx["attrs"].str.extract(r'gene_name "([^"]+)"')[0]
tx = tx.dropna(subset=["gene_name"])
tx = tx[tx["chrom"].isin(MAIN_CHROMS)]

print(f"{len(tx)} transcripts on main chroms (of {len(gtf)} total)")

34951 transcripts on main chroms (of 541299 total)


## Collapse to one row per gene

In [11]:
# outermost start/end across isoforms. a few genes sit on two chromosomes
# (tandem snRNA/histone/Hsp70 copies, mostly) can't give those one span, so they're out

genes = (tx.groupby(["gene_name", "chrom", "strand"], as_index=False)
           .agg(tx_start=("start", "min"), tx_end=("end", "max")))

n_loci = genes.groupby("gene_name")["chrom"].transform("size")
dropped = sorted(genes.loc[n_loci > 1, "gene_name"].unique())
genes = genes[n_loci == 1].reset_index(drop=True)

assert genes["gene_name"].is_unique

print(f"{len(genes)} genes, single locus")
print(f"dropped {len(dropped)}: {dropped}")
print(genes["chrom"].value_counts().to_string())

16846 genes, single locus
dropped 18: ['7SLRNA:CR32864', '7SLRNA:CR42652', 'CG12491', 'CG14452', 'CG30031', 'CG34105', 'Hsp70Aa', 'Hsp70Ab', 'gammaTry', 'mod(mdg4)', 'snRNA:U1:21D', 'snRNA:U1:95Ca', 'snRNA:U1:95Cb', 'snRNA:U2:34ABb', 'snRNA:U2:34ABc', 'snRNA:U2:38ABb', 'snoRNA:U3:54Aa', 'snoRNA:U3:54Ab']
chrom
chr3R    4064
chr2L    3401
chr2R    3358
chr3L    3355
chrX     2564
chr4      104


## Save

In [12]:
OUT = PROC / "gene_coordinates.csv"
genes.to_csv(OUT, index=False)
print(f"saved -> {OUT.relative_to(ROOT)}")

saved -> data\processed\gene_coordinates.csv
